## Baseline - Naive Bayes (CountVectorizer)

### 📋 Общее описание проекта

Этот блокнот реализует анализ тональности текста с помощью наивного байесовского классификатора. Вот основные этапы работы:

- **Загрузка и предобработка данных**: Используется датасет Sentiment140 с твитами, где целевая переменная кодируется как 0 (негатив) и 1 (позитив)
- **Векторизация текста**: Применяется CountVectorizer для преобразования текста в числовые признаки с параметрами:
  - Размер словаря: 200,000 самых частых слов
  - Минимальная частота слова: 5 раз
  - Максимальная частота: 95% документов
  - N-граммы: от 1 до 2 слов
  - Бинарное кодирование
- **Реализация наивного байесовского классификатора**: Рассчитываются априорные вероятности классов и условные вероятности слов с применением сглаживания Лапласа
- **Предсказание классов**: Для каждого текста вычисляются логарифмы вероятностей для обоих классов и выбирается класс с максимальной вероятностью

### 🎯 Метод предсказания класса

**Наивный байесовский классификатор с сглаживанием Лапласа**

Основные преимущества метода:

- **Высокая скорость работы**: Модель обучается и делает предсказания очень быстро (31 сек на 1.2M+ твитов)
- **Эффективная работа с текстовыми данными**: Хорошо работает с разреженными матрицами признаков
- **Простота реализации**: Легко понять и адаптировать под конкретные задачи
- **Хорошая интерпретируемость**: Можно анализировать, какие слова влияют на предсказание
- **Устойчивость к шуму**: Сглаживание Лапласа предотвращает проблемы с нулевыми вероятностями

### 📊 Результаты модели

| Метрика | Значение |
|---------|----------|
| **Accuracy** | 0.7992 (79.92%) |
| **Precision (негатив)** | 0.79 |
| **Recall (негатив)** | 0.81 |
| **F1-score (негатив)** | 0.80 |
| **Precision (позитив)** | 0.81 |
| **Recall (позитив)** | 0.79 |
| **F1-score (позитив)** | 0.80 |

### 🔍 Примеры работы

**Позитивный текст**: "I am very happy today!"
- Результат: Позитивное настроение
- Уверенность: 96.30%

**Негативный текст**: "I feel so bad and tired"  
- Результат: Негативное настроение
- Уверенность: 99.98%

Модель демонстрирует хорошую точность (около 80%) и уверенность в предсказаниях, что делает ее эффективной для базового анализа тональности текста.

In [2]:
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import classification_report, accuracy_score

# Загружаем датасет
feature_names = ['target', 'id', 'date', 'flag', 'user', 'text']
df = pd.read_csv('./data/Sentiment140.csv', sep=',', encoding='latin-1', names=feature_names)

start = time.time()

# Кодирует целевую переменную: 0 → 0 негатив, 4 → 1 позитив
df['target'] = df['target'].map({0: 0, 4: 1})

# Разделение на трейн и тест
X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["target"], stratify=df["target"], test_size=0.3, random_state=1
)

# Иниц. класса словарь
count_vectorizer = CountVectorizer(
    max_features=200_000, # Размер словаря | 100_000, 150_000, 200_000
    min_df=5, # мин. кол-во встречаемости слова
    max_df=0.95, # макс. встречаемость слова
    ngram_range=(1, 2), # отдельные слова + пары слов
    binary=True #
)

# Получаем матрицу (кол-во сообщений, вектор частоты втречаемости слова)
training_data = count_vectorizer.fit_transform(X_train)
test_data = count_vectorizer.transform(X_test)
print(f"Матрица train: {training_data.shape}")
print(f"Матрица test: {test_data.shape}")
print(f"Размер словаря: {len(count_vectorizer.vocabulary_)}")

# Получаем матрицу строк являющихся положительным классом
good_mask = (y_train == 1).values
sad_mask = (y_train == 0).values

# Получаем априорную веротяность, что сообщение c хорошим настроенияем
P_good = good_mask.sum() / len(y_train)
P_sad = sad_mask.sum() / len(y_train)
print(f"Априорную веротяность позитивного класса: {P_good:.4f}")
print(f"Априорную веротяность негативного класса: {P_sad:.4f}")
print()

# Как часто слово встречается в позитивном классе: P(слово | good)
good_data = training_data[good_mask]
sad_data = training_data[sad_mask]

# Сколько раз слова встретились в положит. / негативных текстах
good_word_counts = np.asarray(good_data.sum(axis=0)).ravel()
sad_word_counts = np.asarray(sad_data.sum(axis=0)).ravel()

# Кол-во положит. / негативных слов
good_word_all = good_word_counts.sum()
sad_word_all = sad_word_counts.sum()

# === Сглаживание Лапласа ===
vocab_size = training_data.shape[1]
print(f"Размер словаря для сглаживания: {vocab_size}")

# Вектор частотности: Сглаженные векторы
alpha = 0.1 # 0.1, 0.5, 0.1

P_word_good = (good_word_counts + alpha) / (good_word_all + alpha * vocab_size)
P_word_sad = (sad_word_counts + alpha) / (sad_word_all + alpha * vocab_size)
# P_word_good = (good_word_counts + alpha) / (good_word_all + vocab_size)
# P_word_sad = (sad_word_counts + alpha) / (sad_word_all + vocab_size)

# Логарифмы вероятностей
log_P_good = np.log(P_good)
log_P_sad = np.log(P_sad)
log_P_word_good = np.log(P_word_good)
log_P_word_sad = np.log(P_word_sad)

# === Считаем score для всех тестовых сообщений ===
score_good = log_P_good + test_data @ log_P_word_good
score_sad = log_P_sad + test_data @ log_P_word_sad

# === Предсказание ===
y_pred = (score_good > score_sad).astype(int)

print(classification_report(y_test, y_pred))
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f} \n")
print(f"Время выполнения: {time.time() - start:.2f} сек.")

Матрица train: (1120000, 200000)
Матрица test: (480000, 200000)
Размер словаря: 200000
Априорную веротяность позитивного класса: 0.5000
Априорную веротяность негативного класса: 0.5000

Размер словаря для сглаживания: 200000
              precision    recall  f1-score   support

           0       0.79      0.81      0.80    240000
           1       0.81      0.79      0.80    240000

    accuracy                           0.80    480000
   macro avg       0.80      0.80      0.80    480000
weighted avg       0.80      0.80      0.80    480000

Accuracy: 0.7992 

Время выполнения: 31.16 сек.


**Результат обучения модели**

Accuracy: 0.7992 

---

In [3]:
import torch

def predict_sentiment(text):
    text_vector = count_vectorizer.transform([text])
    text_array = text_vector.toarray()

    text_tensor = torch.tensor(text_array, dtype=torch.float32)

    # Логарифмы вероятностей классов
    score_good = log_P_good + text_tensor @ log_P_word_good
    score_sad = log_P_sad + text_tensor @ log_P_word_sad

    # Объединяем score
    scores = torch.stack([
        score_sad.squeeze(),
        score_good.squeeze()
    ])

    # Softmax -> вероятности
    probs = torch.softmax(scores, dim=0)

    prob_sad = probs[0].item()
    prob_good = probs[1].item()

    pred = torch.argmax(probs).item()

    print(f"\nТекст: {text}")

    if pred == 1:
        print("Решение: Позитивное настроение")
        print(f"Уверенность: {prob_good:.2%}")
    else:
        print("Решение: Негативное настроение")
        print(f"Уверенность: {prob_sad:.2%}")

    print(f"P(negative) = {prob_sad:.4f}")
    print(f"P(positive) = {prob_good:.4f}")

In [4]:
result = predict_sentiment("I am very happy today!")
print(result)


Текст: I am very happy today!
Решение: Позитивное настроение
Уверенность: 96.30%
P(negative) = 0.0370
P(positive) = 0.9630
None


/var/folders/gs/rwqhgtg15x3f_w7srr79hfk00000gn/T/ipykernel_19747/2930078154.py:10: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  score_good = log_P_good + text_tensor @ log_P_word_good
/var/folders/gs/rwqhgtg15x3f_w7srr79hfk00000gn/T/ipykernel_19747/2930078154.py:11: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  score_sad = log_P_sad + text_tensor @ log_P_word_sad


In [5]:
result = predict_sentiment("I feel so bad and tired")
print(result)


Текст: I feel so bad and tired
Решение: Негативное настроение
Уверенность: 99.98%
P(negative) = 0.9998
P(positive) = 0.0002
None


/var/folders/gs/rwqhgtg15x3f_w7srr79hfk00000gn/T/ipykernel_19747/2930078154.py:10: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  score_good = log_P_good + text_tensor @ log_P_word_good
/var/folders/gs/rwqhgtg15x3f_w7srr79hfk00000gn/T/ipykernel_19747/2930078154.py:11: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  score_sad = log_P_sad + text_tensor @ log_P_word_sad
